In [8]:
# 📓 Jupyter Notebook: Preprocess e1-n0 dataset

import pandas as pd
import numpy as np

# === Step 1: Load CSV ===
df = pd.read_csv("garden.csv")  # 改成你的檔案路徑

# === Step 2: 基本清理 ===
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts')

# === Step 3: 設定環境 ===
df['env'] = 1

# === Step 4: 篩選 e1-n0 ===
df_n0 = df[df['device'] == 'RIOT-BLE-0'].copy()

# === Step 5: 差分 ===
df_n0['rssi_diff'] = df_n0['rssi'].diff()

# === Step 6: normalization (min-max) ===
y_min = df_n0['rssi_diff'].min()
y_max = df_n0['rssi_diff'].max()
df_n0['rssi_norm'] = (df_n0['rssi_diff'] - y_min) / (y_max - y_min)

# === Step 7: 建立 timeslot（每100個封包一段）===
# 先確保順序正確
df_n0 = df_n0.sort_values('ts').reset_index(drop=True)
# 每100筆為一個 timeslot
WINDOW_SIZE = 100
df_n0['timeslot'] = df_n0.index // WINDOW_SIZE
# === Step 8: 統一欄位名稱 ===
df_n0['node'] = 'n0'
df_n0_final = df_n0[['timeslot', 'env', 'node', 'rssi_norm']].rename(columns={'rssi_norm': 'rssi'})

# === Step 9: 顯示結果 ===
df_n0_final.head(101)

,timeslot,env,node,rssi
0,0,1,n0,NaN
1,0,1,n0,0.666667
2,0,1,n0,0.481481
3,0,1,n0,0.592593
4,0,1,n0,0.370370
...,...,...,...,...
96,0,1,n0,0.518519
97,0,1,n0,0.518519
98,0,1,n0,0.370370
99,0,1,n0,0.481481


In [34]:
import pandas as pd
import numpy as np
import glob

# === 參數設定 ===
WINDOW_SIZE = 100
STRIDE = 50

# === 檔案路徑 ===
files = [
    "e0-bridge.csv",
    "e1-lake.csv",
    "e2-forest.csv",
    "e3-river.csv",
    "e4-garden.csv"
]

# === device mapping ===
device_to_label = {
    "RIOT-BLE-0": 0,
    "RIOT-BLE-1": 1,
    "RIOT-BLE-2": 2,
    "RIOT-BLE-3": 3
}

# === 儲存結果 ===
X = []
y = []

# === 處理每個環境 ===
for env_id, file in enumerate(files):

    df = pd.read_csv(file)

    # 時間排序（保險）
    df['ts'] = pd.to_datetime(df['ts'])
    df = df.sort_values('ts')

    # 每個 node 分開處理
    for device, label in device_to_label.items():

        df_node = df[df['device'] == device].copy()
        if len(df_node) < WINDOW_SIZE:
            continue

        # === 差分 ===
        df_node['rssi_diff'] = df_node['rssi'].diff()
        
        # === drop NaN ===
        df_node = df_node.dropna(subset=['rssi_diff'])
        # === normalization（每個 node 各自做）===
        y_min = df_node['rssi_diff'].min()
        y_max = df_node['rssi_diff'].max()

        if y_max - y_min == 0:
            continue

        df_node['rssi_norm'] = (df_node['rssi_diff'] - y_min) / (y_max - y_min)

        # === 轉成 numpy ===
        data = df_node['rssi_norm'].values

        # === 切 sequence ===
        for i in range(0, len(data) - WINDOW_SIZE, STRIDE):
            seq = data[i:i+WINDOW_SIZE]

            X.append(seq)
            y.append(label)

# === 轉 numpy ===
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (6779, 100)
y shape: (6779,)


In [38]:
# =========================
# Step 1: reshape for 1D CNN
# =========================
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# X: (6779, 100)
# y: (6779,)

X = X.astype(np.float32)
y = y.astype(np.int64)

# PyTorch Conv1d input: (batch, channels, length)
X = X[:, np.newaxis, :]   # -> (6779, 1, 100)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("X_train:", X_train_tensor.shape)
print("X_test :", X_test_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("y_test :", y_test_tensor.shape)

X_train: torch.Size([5084, 1, 100])
X_test : torch.Size([1695, 1, 100])
y_train: torch.Size([5084])
y_test : torch.Size([1695])


In [39]:
# =========================
# Step 2: define 1D CNN model
# =========================
import torch.nn as nn

class CNN1D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=5, padding=2),#feature=1因種類只有rssi
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # 100 -> 50

            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # 50 -> 25

            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)    # 25 -> 12
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


num_classes = len(np.unique(y))
model = CNN1D(num_classes=num_classes)
print(model)

CNN1D(
  (features): Sequential(
    (0): Conv1d(1, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1536, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [40]:
# =========================
# Step 3: training setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [41]:
# =========================
# Step 4: training loop
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            loss = criterion(outputs, yb)

            total_loss += loss.item() * xb.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return total_loss / total, correct / total

In [42]:
# =========================
# Step 5: run training
# =========================
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Epoch [1/20] Train Loss: 1.3160, Train Acc: 0.3462 | Test Loss: 1.2413, Test Acc: 0.3735
Epoch [2/20] Train Loss: 1.2021, Train Acc: 0.3975 | Test Loss: 1.1478, Test Acc: 0.4236
Epoch [3/20] Train Loss: 1.1348, Train Acc: 0.4502 | Test Loss: 1.1882, Test Acc: 0.4112
Epoch [4/20] Train Loss: 1.1275, Train Acc: 0.4508 | Test Loss: 1.0694, Test Acc: 0.4903
Epoch [5/20] Train Loss: 1.0626, Train Acc: 0.4851 | Test Loss: 1.0845, Test Acc: 0.4431
Epoch [6/20] Train Loss: 1.0586, Train Acc: 0.4807 | Test Loss: 1.0483, Test Acc: 0.4590
Epoch [7/20] Train Loss: 1.0247, Train Acc: 0.4923 | Test Loss: 1.0889, Test Acc: 0.4360
Epoch [8/20] Train Loss: 1.0124, Train Acc: 0.5142 | Test Loss: 0.9739, Test Acc: 0.5805
Epoch [9/20] Train Loss: 1.0332, Train Acc: 0.5124 | Test Loss: 0.9809, Test Acc: 0.5853
Epoch [10/20] Train Loss: 0.9677, Train Acc: 0.5610 | Test Loss: 0.9448, Test Acc: 0.5917
Epoch [11/20] Train Loss: 0.9418, Train Acc: 0.5718 | Test Loss: 0.8664, Test Acc: 0.6383
Epoch [12/20] Train

In [43]:
# =========================
# Step 6: prediction example
# =========================
model.eval()

with torch.no_grad():
    sample_x = X_test_tensor[:5].to(device)
    outputs = model(sample_x)
    preds = outputs.argmax(dim=1).cpu().numpy()

print("Pred:", preds)
print("True:", y_test[:5])

Pred: [0 1 0 0 1]
True: [0 0 0 0 0]


In [44]:
# =========================
# Step 7: confusion matrix
# =========================
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_true.extend(yb.numpy())

cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(all_true, all_preds))

Confusion Matrix:
 [[318  75  48  50]
 [ 26 355  24   3]
 [  4  82 294   7]
 [  0  40 100 269]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.65      0.76       491
           1       0.64      0.87      0.74       408
           2       0.63      0.76      0.69       387
           3       0.82      0.66      0.73       409

    accuracy                           0.73      1695
   macro avg       0.75      0.73      0.73      1695
weighted avg       0.76      0.73      0.73      1695

